In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# 1. Linear Models
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC

# 2. Distance-Based
from sklearn.neighbors import KNeighborsClassifier

# 3. Tree-Based
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 4. Advanced Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
# Sample dataset (replace with your actual data/dataframe)
df=pd.read_csv(r"C:\Users\SOLOMON\Downloads\dataset.csv")

In [2]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
df[df["Category"]=="spam"]

,Category,Message
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
5,spam,FreeMsg Hey there darling it's been 3 week's n...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...
11,spam,"SIX chances to win CASH! From 100 to 20,000 po..."
...,...,...
5537,spam,Want explicit SEX in 30 secs? Ring 02073162414...
5540,spam,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,spam,Had your contract mobile 11 Mnths? Latest Moto...
5566,spam,REMINDER FROM O2: To get 2.50 pounds free call...


In [2]:

# Split data
le = LabelEncoder()
y_encoded = le.fit_transform(df['Category']) 

# Split Data
X_train, X_test, y_train, y_test = train_test_split(
    df['Message'], y_encoded, test_size=0.25, random_state=42
)
# Vectorizers
vectorizers = {
    'CountVectorizer': CountVectorizer(),
    'TfidfVectorizer': TfidfVectorizer()
}

# The 9 exact classifiers requested
classifiers = {
    'Perceptron': Perceptron(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000,random_state=42),
    'Linear SVM': LinearSVC(max_iter=10000,random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'Neural Network (MLP)': MLPClassifier(max_iter=500, random_state=42)
}

results = []

best_accuracy = -1.0
best_model = None
best_vectorizer = None
best_model_name = ""
best_vec_name = ""

print("--- Benchmark Running Across 18 Combinations ---\n")

for vec_name, vectorizer in vectorizers.items():
    # Fit vectorizer and transform text data
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    # LightGBM requires float32/64 sparse or dense array
    X_train_vec_f32 = X_train_vec.astype(np.float32)
    X_test_vec_f32 = X_test_vec.astype(np.float32)

    for clf_name, clf in classifiers.items():
        # Pass float32 matrices for LightGBM compatibility
        if clf_name == 'LightGBM':
            clf.fit(X_train_vec_f32, y_train)
            preds = clf.predict(X_test_vec_f32)
        else:
            clf.fit(X_train_vec, y_train)
            preds = clf.predict(X_test_vec)

        acc = accuracy_score(y_test, preds)
        
        results.append({
            'Vectorizer': vec_name,
            'Classifier': clf_name,
            'Accuracy': acc
        })

        # Track the top performing model
        if acc > best_accuracy:
            best_accuracy = acc
            best_model = clf
            best_vectorizer = vectorizer
            best_model_name = clf_name
            best_vec_name = vec_name

# Display ranked results
results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print(results_df.to_string(index=False))

print("\n" + "=" * 50)
print(f"Winner: {best_model_name} using {best_vec_name}")
print(f"Top Test Accuracy: {best_accuracy:.4f}")
print("=" * 50)

# Save the best pipeline assets
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(best_vectorizer, 'best_vectorizer.pkl')

print("\nSaved assets:")
print("- 'best_model.pkl'")
print("- 'best_vectorizer.pkl'")

--- Benchmark Running Across 18 Combinations ---

     Vectorizer           Classifier  Accuracy
TfidfVectorizer           Linear SVM  0.980617
TfidfVectorizer Neural Network (MLP)  0.979182
CountVectorizer           Linear SVM  0.977028
CountVectorizer Neural Network (MLP)  0.976310
CountVectorizer           Perceptron  0.976310
CountVectorizer        Random Forest  0.975592
CountVectorizer  Logistic Regression  0.974874
TfidfVectorizer        Random Forest  0.974156
TfidfVectorizer              XGBoost  0.974156
TfidfVectorizer           Perceptron  0.974156
CountVectorizer              XGBoost  0.972721
CountVectorizer             LightGBM  0.972721
TfidfVectorizer             LightGBM  0.972721
TfidfVectorizer        Decision Tree  0.969131
CountVectorizer        Decision Tree  0.966978
TfidfVectorizer  Logistic Regression  0.963388
CountVectorizer                  KNN  0.914573
TfidfVectorizer                  KNN  0.909548

Winner: Linear SVM using TfidfVectorizer
Top Test Accura